# Task 07 — Identification: D3 × E1

**Business problem.** A company randomly hands AI assistants to salespeople. Salespeople share
leads, so a person's performance may depend both on whether *they* got AI and on whether their
*collaborators* mostly got AI. Before estimating anything we ask: under the current experiment
design and assumptions, can these causal quantities be pinned down from the population law?

**Setting (Task 07 = D3 × E1).**

- **D3 (two-group fixed treated quota).** Units are split into two pre-treatment groups `B1`, `B2`.
  Independently and uniformly, exactly `m1` units are treated in `B1` and `m2` in `B2`. Edges may cross groups.
- **E1 (strict-majority exposure).** With `K_i(t) = sum_{j in N(i)} t_j`, `S_i(t) = 1{K_i(t) > floor(d_i/2)}`.

**Target.** The four design-standardized arm means `mu_i(t, s)` for `(t, s) in {0,1}^2`.

This notebook follows: business problem -> setting -> support -> POINT derivation -> IdentificationResult.

## 1. Setting and assignment law

The D3 design induces the known full-vector assignment law

```
q(t) = 1 / (C(|B1|, m1) * C(|B2|, m2))   if  sum_{i in B1} t_i = m1 and sum_{i in B2} t_i = m2,
     = 0                                  otherwise.
```

For a node `i` in group `g`, conditioning on `T_i = t` leaves `m_g - t` treated units to be drawn
uniformly among the remaining `|B_g| - 1` members of group `g`; the other group is drawn independently.
Hence each group's treated-neighbour count is **hypergeometric** (convention `Hypergeom(N, K, n)` =
population, successes, draws):

```
K_own ~ Hypergeom(|B_g| - 1, m_g - t, d_own)        d_own = |N(i) & B_g|
K_oth ~ Hypergeom(|B_oth|,   m_oth,   d_oth)        d_oth = |N(i) & B_oth|
```

and `pi_i(t, s) = P(T_i = t, S_i = s | A, X)`.

In [ ]:
import os, sys

_here = os.getcwd()
if not os.path.exists(os.path.join(_here, "support.py")):
    for _root, _dirs, _files in os.walk(_here):
        if "support.py" in _files:
            _here = _root
            break
sys.path.insert(0, _here)

from fractions import Fraction
from itertools import product
from collections import defaultdict

from support import TwoGroupQuotaDesign

nodes = ["a", "b", "c", "d", "e", "f"]
edges = [("a", "b"), ("a", "c"), ("a", "d"),
         ("d", "e"), ("d", "f"),
         ("b", "e"), ("c", "f")]

design = TwoGroupQuotaDesign(
    node_ids=nodes,
    undirected_edges=edges,
    group1=["a", "b", "c"],
    group2=["d", "e", "f"],
    m1=2,
    m2=2,
)
print("group1 =", design.group1, "m1 =", design.m1)
print("group2 =", design.group2, "m2 =", design.m2)
print("degrees:", {n: design.degree(n) for n in design.node_ids})

## 2. Query-specific population support

`pi_i(t, s)` is computed exactly (rational arithmetic) from the known design - never from observed data.

In [ ]:
print(f"{'node':>4} | " + " | ".join(f"pi({t},{s})" for t in (0, 1) for s in (0, 1)))
for node in design.node_ids:
    table = design.support_table(node)
    row = " | ".join(f"{str(table[(t, s)]):>9}" for t in (0, 1) for s in (0, 1))
    print(f"{node:>4} | {row}")
    assert sum(table.values()) == Fraction(1), "support must sum to one"

print()
print("supported arms (pi > 0):")
for node in design.node_ids:
    print(f"  {node}: {design.supported_arms(node)}")

## 3. Why the supported arms are POINT

Fix a pretreatment network and context `(A, X)`. Assume

1. **Consistency** - `T = t` implies `Y_obs_i = Y_i(t)`;
2. **Full-vector randomization** - `T` is independent of the potential-outcome schedule given `A, X`
   (it holds by the experiment protocol, the known law `q`);
3. **Fixed pretreatment network** and **finite conditional first moments**.

For an arm with `pi_i(t, s) > 0`, starting from the observable conditional mean and inserting `q`:

```
E[Y_obs_i | A,X,T_i=t,S_i=s]
  = sum_{t_-i : S_i((t,t_-i))=s} E[Y_obs_i | A,X,T=(t,t_-i)] * w(t_-i)
  = sum_{t_-i : S_i((t,t_-i))=s} E[Y_i((t,t_-i)) | A,X] * w(t_-i)
  = mu_i(t, s),                       w(t_-i) = q((t,t_-i)) / pi_i(t, s).
```

The right-hand side is a functional of the population observed law `P(T, Y_obs | A, X)`, so the target is
**POINT identified**. If `pi_i(t, s) = 0` the arm is an empty design cell and the identity does not apply:
the query must be refused rather than silently answered.

In [ ]:
import rule

ASSUMED = rule.EXPERIMENTAL_ASSUMPTIONS  # the five confirmed experimental assumptions

for (t, s) in [(1, 0), (0, 1), (1, 1)]:
    result = rule.identify_arm(design, "b", t, s, assumptions=ASSUMED)
    print(f"arm {(t, s)}: {result.status:>11} {result.verification_status:>10} "
          f"{result.workflow_status:>11} | support={result.support} | {result.functional}")

print()
print("estimation_request (non-None only when executable):")
print(rule.identify_arm(design, "b", 1, 0, assumptions=ASSUMED).estimation_request)

print()
print("contrasts for node b:")
for kind in ("Direct", "Spillover", "Total"):
    result = rule.identify_contrast(design, "b", kind, assumptions=ASSUMED)
    print(f"  {kind:>9}: {result.status} {result.verification_status} {result.workflow_status} | {result.functional}")

## 4. Zero support is refused, and assumptions are never silently admitted

For node `a` the arm `(0, 0)` is impossible under D3 (`pi_a(0,0) = 0`): given `T_a = 0`, all of `B1`'s
quota goes to `a`'s two `B1`-neighbours, so `S_a = 1` deterministically. The rule must refuse the query.

The rule also refuses when the assumption set is omitted: omission is **not** read as blanket admission.

In [ ]:
def report(label, result):
    print(f"{label:>24} | {result.status:>11} | {result.verification_status:>10} | "
          f"{result.workflow_status:>11} | {result.reason_code}")

report("arm (0,0) node a", rule.identify_arm(design, "a", 0, 0, assumptions=ASSUMED))
print("   ", rule.identify_arm(design, "a", 0, 0, assumptions=ASSUMED).message)

report("omitted assumptions", rule.identify_arm(design, "b", 1, 0))
report("incomplete assumptions", rule.identify_arm(design, "b", 1, 0, assumptions=["consistency"]))
report("invalid arm (5,0)", rule.identify_arm(design, "b", 5, 0, assumptions=ASSUMED))

## 5. Brute-force oracle on a small graph (N <= 8)

Enumerate every legal full assignment `t`, give it the uniform design weight, and bucket mass by `(T_i, S_i)`.
This is the independent "standard answer" that the analytic support is checked against. It builds its own
adjacency from the raw edge list, so it does not reuse `support.py`.

In [ ]:
from math import comb

def brute_force_support(node_ids, edges, group1, group2, m1, m2):
    index = {n: k for k, n in enumerate(node_ids)}
    adjacency = defaultdict(set)
    for u, v in edges:
        adjacency[u].add(v)
        adjacency[v].add(u)
    legal = [a for a in product((0, 1), repeat=len(node_ids))
             if sum(a[index[j]] for j in set(group1)) == m1
             and sum(a[index[j]] for j in set(group2)) == m2]
    weight = Fraction(1, len(legal))
    acc = defaultdict(Fraction)
    for a in legal:
        for n in node_ids:
            K = sum(a[index[j]] for j in adjacency[n])
            exp = int(K > len(adjacency[n]) // 2)
            acc[(n, a[index[n]], exp)] += weight
    return acc, len(legal)

oracle, n_legal = brute_force_support(nodes, edges, design.group1, design.group2, design.m1, design.m2)
ok = all(design.support(n, t, s) == oracle.get((n, t, s), Fraction(0))
         for n in nodes for t in (0, 1) for s in (0, 1))
print("legal assignments:", n_legal, "== C(3,2)*C(3,2) =", comb(3, 2) * comb(3, 2))
print("analytic support == brute-force oracle:", ok)
assert ok and n_legal == comb(3, 2) * comb(3, 2)

## 6. IdentificationResult summary

- **Target:** `mu_i(t, s) = E[Y_obs_i | A, X, T_i=t, S_i=s]`.
- **Assumptions:** known assignment law, full-vector randomization, consistency, fixed pretreatment
  network, finite conditional first moments - all passed **explicitly** to the rule.
- **Support:** `pi_i(t, s) > 0` computed exactly from the D3 law; `pi_i(t, s) = 0` => query refused.
- **Result surface (same vocabulary as the shared repo demo):**
  `status in {POINT, UNSUPPORTED}`, `verification_status in {VERIFIED, REJECTED}`,
  `diagnostics["workflow_status"] in {COMPLETE, NEEDS_INPUT, UNSUPPORTED}`, and
  `estimation_request is not None` only when the target is executable.
- **Conclusion:** every positive-support arm is POINT identified; the independent brute-force oracle
  agrees with the analytic support on the small graph.
- **Scope:** Identification only - no estimation, no policy, and no use of any simulated outcome truth.

> Same POINT family as `notebooks/Shared_ER_Identification_Estimation_Policy_Demo.ipynb`.
> The difference is the assignment design: the shared demo uses D1 (`K_i ~ Binomial(d_i, 1/2)`),
> while Task 07 uses D3 (a convolution of two hypergeometrics), which is why an arm can have zero support here.